In [ ]:
# Importing Libraries
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.io import read_image
import torchvision.transforms.functional as F_t
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets.utils import download_url
import matplotlib.pyplot as plt
from IPython import display
import cv2
from PIL import Image
import numpy as np
import os


try:
    from einops import rearrange
except ImportError:
    %pip install einops
    from einops import rearrange

In [ ]:
class RFF(nn.Module): # Random Fourier Features
    def __init__(self, in_features, out_features, gamma): # gamma is the scaling factor
        super().__init__() # super() allows us to refer to the parent class without naming it explicitly
        self.B = torch.randn(in_features, out_features)*gamma # Randomly initialize the weights

    def forward(self, x): # Forward pass
        return torch.hstack([torch.sin(torch.matmul(2*math.pi*x, self.B)), torch.cos(torch.matmul(2*math.pi*x, self.B))]) # Compute the random features

In [ ]:
class RFF_SR(nn.Module): # Random Fourier Features for Super-Resolution
    def __init__(self, in_features, fourier_features, out_features, gamma): # gamma is the scaling factor
        super().__init__() # super() allows us to refer to the parent class without naming it explicitly
        self.net = nn.Sequential(RFF(in_features, fourier_features, gamma), nn.Linear(2*fourier_features, out_features)) 
        
    def forward(self, x): # Forward pass
      return self.net(x) # Compute the random features

In [ ]:
def create_coord_map2x(image_path): # Function to create the coordinate map for the input image
        image = read_image(image_path) # Read the image
        h, w = image.shape[1]*2, image.shape[2]*2 # Get the height and width of the image
        image = image.float()/255 # Normalize the image
        image = image.permute(1, 2, 0) # Permute the dimensions of the image
        # print(image)

        h_coords = torch.linspace(0, 1, steps=h) # Create the height coordinates
        w_coords = torch.linspace(0, 1, steps=w) # Create the width coordinates
        grid = torch.stack(torch.meshgrid(h_coords, w_coords), dim=-1) # Create the grid
        # print(h, w)

        return grid, image # Return the grid and the image

In [ ]:
img_init = cv2.imread("dog1.jpg") # Read the image
img_init = cv2.cvtColor(img_init, cv2.COLOR_BGR2RGB) # Convert the image from BGR to RGB
plt.imshow(img_init) # Display the image
print(img_init.shape) # Print the shape of the image

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Set the device
img_path = "dog1.jpg" # Set the image path
grid, image = create_coord_map2x(img_path) # Create the coordinate map
print(grid.shape, image.shape) # Print the shape of the grid and the image
grid, image = grid.squeeze().to(device), image.squeeze().to(device) # Send the grid and the image to the device
train_coords, train_rgbs = grid[::2, ::2].reshape(-1, 2), image[::, ::].reshape(-1, 3) # Get the training coordinates and RGB values
print(train_coords.shape, train_rgbs.shape) # Print the shape of the training coordinates and RGB values
test_coords, test_rgbs = grid.reshape(-1, 2), image.reshape(-1, 3) # Get the test coordinates and RGB values
print(test_coords.shape, test_rgbs.shape) # Print the shape of the test coordinates and RGB values
train_rgbs_3d = torch.reshape(train_rgbs, (100, 100, 3)) # Reshape the training RGB values


In [ ]:
train_coords

In [ ]:
test_coords

In [ ]:
train_rgbs, train_rgbs.shape

In [ ]:
train_rgbs_np = train_rgbs_3d.cpu().numpy() # Convert the training RGB values to a NumPy array
print(train_rgbs_np.shape) # Print the shape of the training RGB values
plt.imshow(train_rgbs_np) # Display the training RGB values

In [ ]:
SuperReso = RFF_SR(2, 5000, 3, gamma=10).to(device) # Create the Super-Resolution model
optimizer = torch.optim.Adam(SuperReso.parameters(), lr=0.01) # Create the optimizer
train_mse_arr = [] # Create an empty list to store the training MSE loss
train_psnr_arr = [] # Create an empty list to store the training PSNR
test_mse_arr = [] # Create an empty list to store the test MSE loss
test_psnr_arr = [] # Create an empty list to store the test PSNR

for iter in range(1, 101): # Iterate through 100 epochs
    SuperReso.train() # Set the model to training mode
    optimizer.zero_grad() # Zero the gradients
    output = SuperReso(train_coords) # Get the output
    train_loss = F.mse_loss(output, train_rgbs) # Compute the MSE loss
    train_loss.backward() # Backpropagate the loss
    optimizer.step() # Update the weights
    train_mse_arr.append(train_loss.item()) # Append the training MSE loss to the list
    train_psnr_arr.append(10*torch.log10(255*255/train_loss)) # Append the training PSNR to the list

    with torch.no_grad(): # No need to track the gradients
        prediction = SuperReso(test_coords) # Get the prediction
        print(f"At iteration {iter}, the PSNR is {-10*torch.log10(train_loss):.6f} and MSE loss is {train_loss.item():.6f}") # Print the PSNR and MSE loss
    if iter % 2 == 0: # If the iteration is divisible by 2
        fig, axes = plt.subplots(1, 2, figsize=(15, 5)) # Create a figure and axes
        axes[0].imshow(prediction.reshape(image.shape[0]*2, image.shape[1]*2, image.shape[2]).cpu()) # Display the prediction
        axes[0].set_title(f"Prediction Result in Iteration {iter}") # Set the title
        axes[1].imshow(image.cpu()) # Display the original image
        axes[1].set_title(f"Original Image") # Set the title
        plt.show() # Show the plot

In [ ]:
plt.plot(train_mse_arr) # Plot the training MSE loss
plt.xlabel("Number of Iterations") # Set the x-axis label
plt.ylabel("MSE Loss") # Set the y-axis label
plt.title("MSE Loss vs Number of Iterations during Training") # Set the title
plt.show()  # Show the plot

In [ ]:
for i in range(len(train_psnr_arr)): # Iterate through the training PSNR list
    train_psnr_arr[i] = train_psnr_arr[i].item() # Convert the tensor to a NumPy array

plt.plot(train_psnr_arr) # Plot the training PSNR
plt.xlabel("Number of Iterations") # Set the x-axis label
plt.ylabel("PSNR") # Set the y-axis label
plt.title("PSNR vs Number of Iterations during Training") # Set the title
plt.show() # Show the plot